# 05 · Nemori — 실제 대화로 동작 확인

Nemori는 "무엇을 기억할지"를 **prediction error**로 정한다 — 기존 지식이
예측하지 못한 것만 semantic memory가 된다. 두 모듈이 cascade로 이어진다:

```
Episodic Memory Integration (§3.2)   발화 버퍼 → partition → 서사 episode → 병합
Semantic Knowledge Distillation (§3.3)   evoke한 지식으로 예측 → gap만 fact로 추출
```

실제 LoCoMo 대화(conv-26 세션 1~4)를 넣고 확인한다:

- 윈도우 하나에 섞인 두 세션을 partition이 가르는 장면
- 대화가 3인칭 서사 + 시간 앵커로 바뀌는 것
- 윈도우 경계로 찢긴 사건이 병합(supersede)으로 도로 붙는 것
- evoke의 τ 경계 — 예측 재료로 물려오는 지식과 차단되는 지식
- QA에서 temporal 체인("yesterday" → 절대 날짜)이 작동하는 것

구현: `src/memlab/methods/nemori/` (논문 arXiv 2508.03341v4가 명세서.
해석·결정은 각 파일 docstring과 검증 리뷰 N1~N17 참고).

## 준비 — 부품 조립

아래 셀이 하는 일:

1. conv-26을 로드한다 (Caroline ↔ Melanie, 19세션)
2. LM Studio에 연결한다 (`llm.calls`/`llm.total_tokens`로 비용 자동 집계)
3. `NemoriMethod`를 만든다 — 저장소가 인메모리라 인스턴스 생성이 곧 실험
   격리다 (zep의 DB wipe에 해당하는 것이 없다)

config 기본값은 논문 §4.1의 기호 그대로: w=20(observation window),
Ke=5(병합 후보), Ks=10(evoke 회수량), k=10·m=20·r=2(답변 검색).
τ만 0.70→0.55 — 논문 값은 openai embedding 공간 기준이라 우리 all-MiniLM
공간에서 실측 재캘리브레이션했다 (근거는 method.py docstring, 검증 리뷰 N2).

관찰에 쓰는 `method._d_e`(episodic DB)·`method._d_s`(semantic DB)·
`method._buffer`는 내부 상태다 — 가이드에서만 들여다본다.

In [1]:
from memlab.data import load_locomo
from memlab.evaluation import set_f1
from memlab.llm import default_provider
from memlab.methods import Utterance
from memlab.methods.nemori import NemoriMethod

sample = load_locomo()[0]
sessions = {s.index: s for s in sample.sessions}

llm = default_provider()
method = NemoriMethod(llm)

def ingest_session(session):
    for t in session.turns:
        method.ingest(Utterance(t.speaker, t.text, session.date_time, t.blip_caption))
    print(f"세션 {session.index} ({session.date_time}) — 발화 {len(session.turns)}개"
          f" / 누적 LLM {llm.calls}회 / buffer 대기 {len(method._buffer)}개")

print(f"{sample.sample_id}: {sample.speaker_a} ↔ {sample.speaker_b}, 세션 {len(sample.sessions)}개")

conv-26: Caroline ↔ Melanie, 세션 19개


## 1. Partitioning — 버퍼가 episode로 갈린다 (§3.2.1)

발화는 일단 버퍼에 쌓이고, **w=20개가 차는 순간** LLM이 topic 단위로
partition한다 (같은 입력이면 항상 같은 경계 — 원본 구현의 async 타이밍
비결정성을 제거한 지점). 세션 1이 18발화라, 첫 윈도우에는 세션 1 전체와
세션 2의 첫 2발화가 **섞여 들어간다** — partition이 이걸 어떻게 다루는지가
관전 포인트다.

In [2]:
ingest_session(sessions[1])
ingest_session(sessions[2])

print(f"\nD_e — episode {len(method._d_e.items)}개:")
for ep in method._d_e.items:
    print(f"  [{ep.occurred_at}] ({len(ep.raw):2d}발화) {ep.cue}")

uuids_after_s2 = {ep.uuid for ep in method._d_e.items}  # 3절의 supersede 관찰용

세션 1 (1:56 pm on 8 May, 2023) — 발화 18개 / 누적 LLM 0회 / buffer 대기 18개


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

세션 2 (1:14 pm on 25 May, 2023) — 발화 17개 / 누적 LLM 6회 / buffer 대기 15개

D_e — episode 2개:
  [2023-05-08 13:56:00] (18발화) May 2023 Check-in: LGBTQ Support, Career Goals in Counseling, and Painting Sharing
  [2023-05-25 13:14:00] ( 2발화) May 25, 2023: Melanie shares mental health charity race experience


출력에서 볼 것:

- **세션 경계 분리** — 5월 8일 episode와 5월 25일 episode가 따로 생겼다면,
  한 윈도우에 섞인 두 세션을 PARTITION 프롬프트의 "30분 이상 시간 갭" 신호가
  갈라낸 것이다
- **occurred_at** — 벽시계가 아니라 LLM이 대화에서 추출한 시각. 세션
  date_time과 분 단위까지 일치하는지 확인
- buffer 대기 수 — 세션 2가 끝나도 20개 미만 잔여는 버퍼에 남아 있다
  (다음 윈도우 또는 `end_ingest()`가 처리)

## 2. Narrative — 대화가 서사와 시간 앵커로 바뀐다 (§3.2.2)

partition된 그룹마다 NARRATIVE 프롬프트가 (episodic_cue, narrative_episode,
timestamp)를 한 번에 뽑는다. 원문(raw)은 버리지 않고 episode에 그대로
보관된다 — 답변 시 상위 r=2 episode는 원문까지 첨부되는 dual-mode 설계다.

In [3]:
ep = method._d_e.items[0]
print("cue      :", ep.cue)
print("occurred :", ep.occurred_at)
print("\n-- narrative --")
print(ep.narrative)
print("\n-- raw 원문 (앞 3발화) --")
for u in ep.raw[:3]:
    print(f"  [{u.timestamp}] {u.speaker}: {u.text[:80]}")

cue      : May 2023 Check-in: LGBTQ Support, Career Goals in Counseling, and Painting Sharing
occurred : 2023-05-08 13:56:00

-- narrative --
On May 8, 2023 at 1:56 PM, Caroline and Melanie exchanged messages to catch up. Although both reported being busy with kids and work, they immediately discussed personal updates. Caroline revealed that she had attended an LGBTQ support group the previous day (May 7, 2023), describing it as powerful and noting how transgender stories made her feel happy and thankful. She shared an image of a dog walking past a wall featuring a painting of a woman, which Melanie praised as cool and helpful. The conversation deepened when Caroline explained that the group gave her a sense of acceptance and courage to embrace herself. Looking toward the future, Caroline expressed excitement about continuing her education and exploring career options in mental health counseling to support others with similar issues; Melanie enthusiastically encouraged this path, compl

출력에서 볼 것:

- **3인칭 서사** — 대화체가 "Caroline shared that..."로 재구성됐다
- **상대 → 절대 날짜 변환** — 원문의 "yesterday"류가 서사에서 절대 날짜로
  바뀌었는지 (NARRATIVE 요건 8). temporal QA의 성패가 여기 걸린다.
  temp 0.7이라 이 변환은 확률적이다 — 안 된 곳이 보이면 그게 dry-run에서
  관찰된 편차다
- **cue의 검색성** — 날짜·주제가 든 한 줄 표제. embedding은 f_emb(cue ∥
  narrative)로 만들어지고, §3.3의 예측 단서로도 쓰인다

## 3. Associative Integration — 찢긴 사건이 도로 붙는다 (§3.2.3)

새 episode마다 기존 D_e에서 cosine top-Ke=5 후보를 뽑아 LLM이 병합 여부를
판정한다(SELECT_TARGET, 같은 사건 + 1시간 이내). 병합되면 두 서사를 다시
써서(INTEGRATE) 옛 episode를 **supersede**한다 — 수정이 아니라 삭제 후
재생성이다. 세션 3~4를 마저 넣고 관찰한다. 마지막의 `end_ingest()`는
잔여(<w) 발화를 flush하는 소켓 훅 — 러너가 ingest 루프 직후 부르는 것을
그대로 재현한다 (검증 리뷰 N1).

In [4]:
ingest_session(sessions[3])
ingest_session(sessions[4])
method.end_ingest()

superseded = uuids_after_s2 - {ep.uuid for ep in method._d_e.items}
print(f"\nD_e — episode {len(method._d_e.items)}개 (supersede된 초기 episode {len(superseded)}개):")
for ep in method._d_e.items:
    print(f"  [{ep.occurred_at}] ({len(ep.raw):2d}발화) {ep.cue}")

세션 3 (7:55 pm on 9 June, 2023) — 발화 23개 / 누적 LLM 17회 / buffer 대기 18개


세션 4 (10:37 am on 27 June, 2023) — 발화 18개 / 누적 LLM 55회 / buffer 대기 16개



D_e — episode 12개 (supersede된 초기 episode 0개):
  [2023-05-08 13:56:00] (18발화) May 2023 Check-in: LGBTQ Support, Career Goals in Counseling, and Painting Sharing
  [2023-05-25 13:14:00] ( 2발화) May 25, 2023: Melanie shares mental health charity race experience
  [2023-05-25 13:14:00] ( 5발화) Post-Event Reflection on Self-Care and Summer Camping Plans May 25, 2023
  [2023-05-25 13:14:00] (10발화) May 2023: Caroline Researches LGBTQ+ Inclusive Adoption Agency and Discusses Single-Parent Journey
  [2023-06-09 19:55:00] (11발화) June 9, 2023: Caroline and Melanie Unite to Celebrate Trans Success, Share Personal Courage Stories, and Commit to Mutual Support
  [2023-06-09 19:55:00] ( 2발화) Melanie and Caroline discuss long-term friendship support systems post-breakup on June 9, 2023
  [2023-06-09 19:55:00] ( 2발화) Sharing family waterfall photo and discussing marriage duration on June 9, 2023
  [2023-06-09 19:55:00] ( 2발화) 5th Wedding Anniversary Celebration and Congratulations Received
  [2023-06-09

출력에서 볼 것:

- **supersede 수** — 0이면 이번 실행에선 병합이 없었던 것이다 (판정이
  temp 0.7 LLM이라 확률적 — dry-run에선 charity race 화제 재등장이 병합된
  적도, 안 된 적도 있다). 1 이상이면 세션 1~2 때 있던 episode가 병합으로
  대체됐다는 뜻 — 발화 수가 커진 episode가 그 병합본이다
- **병합 기준의 보수성** — 화제가 비슷해도 다른 사건이면 "new"가 맞다.
  episode 수가 세션당 1~2개 수준이면 과병합 없이 잘 갈린 것

## 4. Predict-Calibrate — 예측이 못 맞춘 것만 기억이 된다 (§3.3)

episode가 확정될 때마다 (cascade):

1. **Evoke** — episode embedding으로 D_s에서 sim > τ=0.55인 지식 top-Ks 회수
2. **Anticipate** — 회수한 지식 + cue만 주고 "무슨 일이 있었을지" 예측
3. **Distill** — 원문 대화와 예측을 대조해 **gap만** fact로 추출

D_s가 비어 있거나 evoke가 빈손이면 예측 없이 직접 추출한다(cold start,
DIRECT_DISTILL). 이미 아는 fact는 예측이 맞춰버리므로 재추출되지 않는다 —
**dedup을 threshold가 아니라 예측이 담당**하는 게 이 설계의 요체다.

아래 셀은 insight 전체와, 마지막 episode 관점의 evoke 경계(어떤 지식이
sim 몇으로 물려오고 어떤 게 차단되는지)를 보여준다.

In [5]:
import numpy as np
from memlab.embedding import cosine_top_k

print(f"D_s — insight {len(method._d_s.items)}개:")
for ins in method._d_s.items:
    print("  -", ins.statement)

ep = method._d_e.items[-1]
q = np.asarray(ep.embedding)
vectors = [np.asarray(i.embedding) for i in method._d_s.items]
print(f"\n마지막 episode({ep.cue[:50]}...) 관점의 evoke 경계 (τ={method._config.tau}):")
for idx in cosine_top_k(vectors, q, k=5):
    v = vectors[idx]
    sim = float(q @ v / (np.linalg.norm(q) * np.linalg.norm(v)))
    mark = "evoke " if sim > method._config.tau else "차단  "
    print(f"  [{mark}] {sim:.3f}  {method._d_s.items[idx].statement[:70]}")

D_s — insight 41개:
  - The user (Caroline) is a parent with young children.
  - The user has a strong affinity for and identifies with the LGBTQ+ community, specifically expressing happiness regarding transgender stories.
  - The user attended an LGBTQ support group in May 2023 and found it a powerful source of acceptance.
  - The user is pursuing education to become a mental health counselor with a specialization in supporting LGBTQ+ individuals.
  - Melanie prioritizes self-care as a necessary condition for effectively caring for her family.
  - Melanie has developed a routine of daily 'me-time' activities, including running, reading, and playing the violin.
  - Melanie has two children.
  - Caroline is a single parent.
  - Caroline's career goal is to become an adoptive mother.
  - Caroline researched LGBTQ+ inclusive adoption agencies as of May 2023.
  - Caroline has been transitioning for three years.
  - Caroline started her transition in 2020 (three years prior to June 9, 2023).

출력에서 볼 것:

- **statement의 자립성** — 문맥 없이 읽히는 현재시제 fact인지. 날짜가 든
  fact("race on May 20, 2023")는 DISTILL을 논문판으로 채택한 효과다
  (원본 코드판은 날짜 금지였다)
- **evoke 경계** — dry-run 실측에서 관련 지식은 0.60~0.65, 무관은 0.55
  아래로 갈렸다. 차단된 항목이 정말 이 episode와 무관한지 눈으로 검증
- 한계도 그대로 보인다: "The user ..."로 시작하는 statement는 화자 binding이
  샌 것(2화자 대화에 user-assistant 가정 프롬프트), "three years ago"류는
  무앵커 상대 시간이다 — 스모크에서 발생률을 정량화한다

## 5. QA — 기억만으로 답한다 (§3.4, Algorithm 2)

질문 embedding으로 episodic top-k=10 + semantic top-m=20을 병렬 회수,
상위 r=2 episode에는 원문을 첨부해 ANSWER 프롬프트로 답을 뽑는다.
answer는 READ 전용이다 — 위에서 `end_ingest()`를 이미 불렀으므로 버퍼는
비어 있다. 근거가 세션 1~4 안에 있는 실제 LoCoMo QA로 확인한다.

In [6]:
covered = {t.dia_id for i in (1, 2, 3, 4) for t in sessions[i].turns}
exam = [q for q in sample.qa if q.answer and q.evidence and set(q.evidence) <= covered][:4]

for qa in exam:
    pred = method.answer(qa.question)
    print(f"Q: {qa.question}")
    print(f"   gold: {qa.answer}  |  예측: {pred.strip()}  |  set_f1 {set_f1(pred, qa.answer):.2f}\n")

print(f"총 비용: LLM {llm.calls}회 / {llm.total_tokens:,} 토큰")

Q: When did Caroline go to the LGBTQ support group?
   gold: 7 May 2023  |  예측: May 7, 2023.  |  set_f1 1.00



Q: When did Melanie paint a sunrise?
   gold: 2022  |  예측: May 2023 (last year)  |  set_f1 0.00



Q: What fields would Caroline be likely to pursue in her educaton?
   gold: Psychology, counseling certification  |  예측: Mental health counseling and adoption.  |  set_f1 0.25



Q: What did Caroline research?
   gold: Adoption agencies  |  예측: LGBTQ+ inclusive adoption agencies.  |  set_f1 0.67

총 비용: LLM 64회 / 83,012 토큰


출력에서 볼 것:

- **temporal 체인** — "When did Caroline go to the LGBTQ support group?"의
  근거 발화는 "yesterday"뿐이다. 세션 날짜(5월 8일)와 결합해 May 7을
  내면 episode의 시간 앵커가 일한 것
- **답변 간결성** — 전부 몇 단어 수준이어야 한다. ANSWER 프롬프트의
  "5-6단어 이내" 지시가 step-by-step 지시를 이긴다 (검증 리뷰에서 실측)
- set_f1이 1.0이 아닌 정답도 있다 — "Last year (2022)" vs gold "2022"처럼
  표현 차이는 감점된다. 카테고리 집계는 스모크·전량 런의 몫

## 정리

conv-26 세션 4개로 cascade 전 경로를 실물로 확인했다:

- partition이 세션 경계를 갈랐고, 서사가 시간 앵커를 갖게 됐고, 병합이
  윈도우로 찢긴 사건을 이었다 (§3.2)
- evoke → 예측 → gap 추출이 τ=0.55 경계에서 작동했다 (§3.3)
- QA가 temporal 체인을 포함해 몇 단어 답으로 돌아왔다 (§3.4)

비용은 [[Zep]]의 1/15 이하 — episode 단위 처리가 message 단위 처리를
피한 효과다 (논문 §4.3의 주장 그대로).

다음: 스모크(conv-26 전체 419발화 + 199문항 채점) → 전량 런 → README.
run 아티팩트는 `runs/nemori-smoke/`, 판정 기록은 obsidian vault의
`memlab/runs/` 노트 참고.